## Linearize FCDF triples for KG-RAG

In this notebook we read a Turtle file, converts each RDF triple into a simple text line, and writes the output to JSONL.

In [1]:
# Install dependency (run once)
!pip install -q rdflib


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json

from rdflib import Graph, URIRef, Literal
from rdflib.namespace import RDF, RDFS

project_root = Path(".").resolve()

ttl_path = project_root / "fcdf_kg.ttl"
output_path = project_root / "fcdf_kg_triples_linearized.jsonl"

In [3]:
project_root

WindowsPath('C:/Users/dyury/Desktop/Master Thesis')

In [4]:
def local_name(uri: str) -> str:
    # Return URI tail after # or last /.
    if "#" in uri:
        return uri.rsplit("#", 1)[-1]
    return uri.rstrip("/").rsplit("/", 1)[-1]

# Entity-centric RAG buckets use these stable id prefixes as focal keys.
SUBJECT_ID_PREFIXES = ("player/", "team/", "referee/")

def get_label(node, g: Graph) -> str:
    # Use rdfs:label when available, otherwise fallback to URI tail/literal text.
    if isinstance(node, Literal):
        return str(node)
    if isinstance(node, URIRef):
        lbl = next(g.objects(node, RDFS.label), None)
        if lbl is not None:
            return str(lbl)
        # :name on Referee (and similar) when no rdfs:label
        nm = next(g.objects(node, URIRef("https://w3id.org/football-cdf/core#name")), None)
        if nm is not None:
            return str(nm)
        return local_name(str(node))
    return str(node)

def get_subject_label(node, g: Graph) -> str:
    # Hybrid (option C): player/team/referee subjects keep URI tails; others unchanged.
    if isinstance(node, URIRef):
        tail = local_name(str(node))
        if tail.startswith(SUBJECT_ID_PREFIXES):
            return tail
    return get_label(node, g)

def linearize_triple(s, p, o, g: Graph) -> str:
    # Subject: stable id for player/team/referee; object: human-readable labels.
    s_label = get_subject_label(s, g)
    p_label = get_label(p, g)
    o_label = get_label(o, g)
    return f"{s_label} {p_label} {o_label}"

In [5]:
# Reads TTL and write linearized triples to JSONL
g = Graph()
g.parse(ttl_path, format="turtle")
print("Triples in graph:", len(g))

count = 0
with open(output_path, "w", encoding="utf-8") as f:
    for s, p, o in g:
        text = linearize_triple(s, p, o, g)
        record = {
            "text": text,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        count += 1

Triples in graph: 2189011


In [6]:
# Preview a few linearized triples (reads output_path; use fcdf_kg_triples_linearized.jsonl when USE_SMALL_SAMPLE is False)
preview_n = 5
with open(output_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= preview_n:
            break
        rec = json.loads(line)
        print(rec["text"])

event/4279d9d6-b511-4e00-a000-02a68505909c x 76.3
event/b75bbd66-e629-486d-a6b3-96b99da5e932 type carry
event/3e5a59a4-1209-4df6-942b-2f3c1baa0ef4 misc_outcome_type unsuccessful
event/8e45854f-734c-4396-ad08-f07cab66e5f9 x_end 45.2
event/67454b2f-3f9b-48f2-bfc9-8b0dd47dd51f type Miscellaneous Event
